# SyftBox Manual E2E Test

This notebook walks through complete end-to-end workflows between a **Data Owner (DO)** and a **Data Scientist (DS)** using SyftBox. Each job section is self-contained: submission, review, execution, and result retrieval.

### Job Overview

| Job | Name | Scenario | Expected Outcome |
|-----|------|----------|------------------|
| A | Pre-Submit Scanner | Code contains `client=ds_client` | Submission blocked by scanner |
| B | Summary Statistics | Standard analysis job | Approved, runs successfully |
| C | Column Names | Leaks sensitive column names | Rejected by DO |
| D | PDF Size | Computes total PDF size | Approved, runs successfully |
| E | Missing Dependency | Uses `scipy` without declaring it | Approved, fails during execution |
| F | Infinite Loop | Runs forever | Approved, killed by timeout |

### Prerequisites
- A `.env` file with: `DO_EMAIL`, `DS_EMAIL`, `TOKEN_DO`, `TOKEN_DS`

---

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

### 1.1 Configuration

Load environment variables and validate that credential files exist.

In [ ]:
import os
from pathlib import Path

# Load .env file
for line in Path(".env").read_text().splitlines():
    if line.strip() and not line.startswith("#"):
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip())

DO_EMAIL = os.environ["DO_EMAIL"]
DS_EMAIL = os.environ["DS_EMAIL"]

do_token_path = Path(os.environ["TOKEN_DO"])
ds_token_path = Path(os.environ["TOKEN_DS"])

assert do_token_path.exists() and ds_token_path.exists()

### 1.2 Login

Log in as both the Data Owner and the Data Scientist. Each client maintains its own local SyftBox folder and syncs independently.

In [ ]:
import syft_client as sc
do_client = sc.login_do(email=DO_EMAIL, token_path=do_token_path)

In [ ]:
ds_client = sc.login_ds(email=DS_EMAIL, token_path=ds_token_path)

### 1.3 Optional — Wipe prior state

Uncomment to delete all local SyftBox data and start fresh.

In [ ]:
# do_client.delete_syftbox()
# ds_client.delete_syftbox()

### 1.4 Establish peer connection

The Data Scientist requests access to the Data Owner, and the Data Owner approves. This is required before any data or jobs can be exchanged.

In [ ]:
ds_client.add_peer(DO_EMAIL)
do_client.load_peers()
do_client.approve_peer_request(DS_EMAIL, peer_must_exist=False)

print("Peer relationship established")

In [ ]:
do_client.peers

### 1.5 Shared helpers

In [ ]:
import tempfile

def write_job_script(code: str) -> str:
    """Write a job script to a temp file and return its path."""
    job_dir = Path(tempfile.mkdtemp(prefix="syft_job_"))
    script = job_dir / "main.py"
    script.write_text(code)
    return str(script)

---

## 2. Data Owner: Create Datasets

The Data Owner creates datasets that will be shared with the Data Scientist. Each dataset has:
- **Mock data** — a public, non-sensitive version that the DS can freely access and prototype against
- **Private data** — the real data that never leaves the DO's machine; only job code can access it at runtime

### 2.1 Create a simple text dataset

In [ ]:
do_client.create_dataset(
    name="testdata",
    mock_path="data/mock.txt",
    private_path="data/private.txt",
    users=[DS_EMAIL],
)

### 2.2 Create a PDF dataset

Generate sample PDFs, then create a dataset with a CSV manifest as mock data and the PDFs as private data.

In [ ]:
!uv run create_test_pdfs.py

In [ ]:
do_client.create_dataset(
    name="pdfdata",
    mock_path="data/manifest.csv",
    private_path="data/PDFs",
    summary="A dataset with some PDFs",
    readme_path="data/readme.md",
    tags=["test", "pdf"],
    users=[DS_EMAIL],
    upload_private=False,
    sync=True,
)

### 2.3 View datasets

In [ ]:
do_client.datasets

In [ ]:
dataset = do_client.datasets.get("testdata", datasite=DO_EMAIL)
dataset

In [ ]:
mock_contents = dataset.mock_files[0].read_text()
print(f"Mock file contents: {mock_contents!r}")

### 2.4 DS: Browse available datasets

In [ ]:
ds_client.datasets

### 2.5 DS: Prototype code against mock data

Before submitting a job, the Data Scientist can prototype their analysis locally using the mock data. The `resolve_dataset_file_path` function returns mock data when run locally, and private data when run as a job on the DO's machine.

> **Note:** Remove the `client=` parameter before submitting as a job — it will resolve automatically on the DO's machine.

In [ ]:
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata", client=ds_client)
df = pd.read_fwf(resolved_path)
print(f"Type of data in use: {df.columns[0]}")

---

## 3. Job E2E Flows

Each subsection below is a self-contained end-to-end flow: DS submits a job, DO reviews and acts on it, and DS retrieves the outcome.

---

### 3.1 Job A: Pre-Submit Scanner Validation

This submission contains `client=ds_client` in the resolver call, which would fail on the DO's machine. The pre-submit scanner should detect this and prompt the user to confirm. Typing "N" (or just pressing Enter) blocks the submission.

In [ ]:
# When prompted, type "N" to block the submission

job_a_code = write_job_script('''import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata", client=ds_client)
df = pd.read_fwf(resolved_path)
print(df.describe())
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_a_code,
    job_name="blocked_client_ds",
)

---

### 3.2 Job B: Summary Statistics (full E2E)

A standard analysis job that reads the dataset and computes summary statistics with a simple dependency (`pandas`). This demonstrates the happy path: submit → approve → run → retrieve results.

#### 3.2.1 DS submits job

In [ ]:
JOB_B_NAME = "summary_stats"

job_b_code = write_job_script('''import os, json
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata")
df = pd.read_fwf(resolved_path)

result = {
    "message": "Summary statistics computed successfully",
    "row_count": len(df),
    "columns": list(df.columns),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_b_code,
    job_name=JOB_B_NAME,
    force_submission=True,
    dependencies=["pandas"],
)
print(f"Job '{JOB_B_NAME}' submitted to {DO_EMAIL}")

#### 3.2.2 DO reviews and approves

In [ ]:
do_client.jobs

In [ ]:
job_b_do = do_client.jobs[JOB_B_NAME]
job_b_do
# print(f"--- {job_b_do.name} (status: {job_b_do.status}) ---")
# print(job_b_do.code)

In [ ]:
job_b_do.approve()

#### 3.2.3 DO runs the job

In [ ]:
do_client.process_approved_jobs(
    share_outputs_with_submitter=True,
    share_logs_with_submitter=True,
)

#### 3.2.4 DS retrieves results

In [ ]:
ds_job_b = ds_client.jobs[JOB_B_NAME]
print(f"Status: {ds_job_b.status}")
print(f"Output files: {ds_job_b.output_paths}")

if ds_job_b.output_paths:
    print(f"\nResult:\n{ds_job_b.output_paths[0].read_text()}")

---

### 3.3 Jobs C & D: Column Names + PDF Size (batch approve)

Two jobs submitted independently, then approved and run together.

- **Job C** (`column_names`) — intentionally leaks sensitive column names; DO will reject it
- **Job D** (`pdf_size_sum`) — computes total PDF file size; DO will approve it

#### 3.3.1 DS submits Job C: Column Names

In [ ]:
JOB_C_NAME = "column_names"

job_c_code = write_job_script('''import os, json
import syft_client as sc
import pandas as pd

resolved_path = sc.resolve_dataset_file_path("testdata")
df = pd.read_fwf(resolved_path)

# This leaks sensitive column names from the private dataset
result = {
    "message": "Extracted column names",
    "column_names": list(df.columns),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_c_code,
    job_name=JOB_C_NAME,
    force_submission=True,
    dependencies=["pandas"],
)
print(f"Job '{JOB_C_NAME}' submitted to {DO_EMAIL}")

#### 3.3.2 DS submits Job D: PDF Size

In [ ]:
JOB_D_NAME = "pdf_size_sum"

job_d_code = write_job_script('''import os, json
import syft_client as sc

dataset_files = sc.resolve_dataset_files_path("pdfdata")

total_bytes = 0
file_count = 0
for file_path in dataset_files:
    size = os.path.getsize(file_path)
    total_bytes += size
    file_count += 1

result = {
    "message": "PDF size computation completed successfully",
    "file_count": file_count,
    "total_bytes": total_bytes,
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_d_code,
    job_name=JOB_D_NAME,
    force_submission=True,
)
print(f"Job '{JOB_D_NAME}' submitted to {DO_EMAIL}")

#### 3.3.3 DO reviews both jobs

In [ ]:
do_client.jobs

In [ ]:
job_c_do = do_client.jobs[JOB_C_NAME]
job_d_do = do_client.jobs[JOB_D_NAME]

print(f"--- {job_c_do.name} ---")
print(f"\n--- {job_d_do.name} ---")

#### 3.3.4 DO rejects C, approves D

In [ ]:
job_c_do.reject(reason="The job leaks sensitive column names from the private dataset")
job_d_do.approve()

#### 3.3.5 DO runs approved jobs

In [ ]:
do_client.process_approved_jobs(
    share_outputs_with_submitter=True,
    share_logs_with_submitter=True,
)

#### 3.3.6 DS retrieves results

In [ ]:
# Job C: should be rejected
ds_job_c = ds_client.jobs[JOB_C_NAME]
print(f"Job C status: {ds_job_c.status}")
print(f"Rejection reason: {ds_job_c.review_reason}")

In [ ]:
# Job D: should have output
ds_job_d = ds_client.jobs[JOB_D_NAME]
print(f"Job D status: {ds_job_d.status}")
print(f"Output files: {ds_job_d.output_paths}")

if ds_job_d.output_paths:
    print(f"\nResult:\n{ds_job_d.output_paths[0].read_text()}")

---

### 3.4 Job E: Missing Dependency (execution failure)

This job uses `scipy` (not a dependency of syft-client) but is submitted **without** `dependencies=["scipy"]`. It will be approved but fail during execution because scipy is not available in the isolated venv.

#### 3.4.1 DS submits job

In [ ]:
JOB_E_NAME = "missing_dependency"

job_e_code = write_job_script('''import os, json
import syft_client as sc
from scipy import stats

resolved_path = sc.resolve_dataset_file_path("testdata")
data = [1, 2, 3, 4, 5]

result = {
    "message": "Scipy stats computed successfully",
    "mean": float(stats.tmean(data)),
    "variance": float(stats.tvar(data)),
}

os.makedirs("outputs", exist_ok=True)
with open("outputs/result.json", "w") as f:
    json.dump(result, f, indent=2)

print(f"Result: {result}")
''')

# Note: no dependencies=["scipy"] — this will cause execution failure
ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_e_code,
    job_name=JOB_E_NAME,
    force_submission=True,
)
print(f"Job '{JOB_E_NAME}' submitted to {DO_EMAIL}")

#### 3.4.2 DO reviews and approves

In [ ]:
job_e_do = do_client.jobs[JOB_E_NAME]
print(f"--- {job_e_do.name} (status: {job_e_do.status}) ---")
# print(job_e_do.code)

In [ ]:
job_e_do.approve()

#### 3.4.3 DO runs the job (will fail)

In [ ]:
do_client.process_approved_jobs(
    share_outputs_with_submitter=True,
    share_logs_with_submitter=True,
)

#### 3.4.4 DS sees the failure

In [ ]:
ds_job_e = ds_client.jobs[JOB_E_NAME]
print(f"Status: {ds_job_e.status}")
print(f"stdout: {ds_job_e.stdout}")
print(f"stderr: {ds_job_e.stderr}")

---

### 3.5 Job F: Infinite Loop (timeout)

A simple job that runs an infinite loop, intended to test the timeout functionality.

#### 3.5.1 DS submits job

In [ ]:
JOB_F_NAME = "infinite_loop"

job_f_code = write_job_script('''import time
while True:
    time.sleep(1)
''')

ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=job_f_code,
    job_name=JOB_F_NAME,
    force_submission=True,
)
print(f"Job '{JOB_F_NAME}' submitted to {DO_EMAIL}")

#### 3.5.2 DO reviews and approves

In [ ]:
job_f_do = do_client.jobs[JOB_F_NAME]
print(f"--- {job_f_do.name} (status: {job_f_do.status}) ---")
# print(job_f_do.code)

In [ ]:
job_f_do.approve()

#### 3.5.3 DO runs the job (will timeout)

In [ ]:
do_client.process_approved_jobs(
    share_outputs_with_submitter=True,
    share_logs_with_submitter=True,
    timeout=10
)

#### 3.5.4 DS sees the timeout

In [ ]:
ds_job_f = ds_client.jobs[JOB_F_NAME]
print(f"Status: {ds_job_f.status}")

In [ ]:
job_f_do.files

---

## 4. Cleanup

Remove generated test files.

In [ ]:
!rm -rf data/PDFs data/manifest.csv data/readme.md

In [ ]:
# do_client.delete_syftbox()
# ds_client.delete_syftbox()